In [3]:
import pandas as pd
import numpy as np
import os

In [4]:
processed_path = "../Dataset/processed"

studentInfo = pd.read_csv(os.path.join(processed_path, "studentInfo_clean.csv"))

studentAssessment = pd.read_csv(os.path.join(processed_path, "studentAssessment_clean.csv"))

studentRegistration = pd.read_csv(os.path.join(processed_path, "studentRegistration_clean.csv"))

studentVle = pd.read_csv(os.path.join(processed_path, "studentVle_clean.csv"))

assessments = pd.read_csv(os.path.join(processed_path, "assessments_clean.csv"))

courses = pd.read_csv(os.path.join(processed_path, "courses_clean.csv"))

vle = pd.read_csv(os.path.join(processed_path, "vle_clean.csv"))

In [5]:
import pandas as pd
import numpy as np
import os
assessment_data = pd.merge(
    studentAssessment,
    assessments,
    on="id_assessment",
    how="left"
)

# Remove final exam (future information)
assessment_data = assessment_data[
    assessment_data["assessment_type"] != "Exam"
]

assessment_data.head()

,id_assessment,id_student,date_submitted,is_banked,score,code_module,code_presentation,assessment_type,date,weight
0,1752,11391,18,0,78.0,AAA,2013J,TMA,19.0,10.0
1,1752,28400,22,0,70.0,AAA,2013J,TMA,19.0,10.0
2,1752,31604,17,0,72.0,AAA,2013J,TMA,19.0,10.0
3,1752,32885,26,0,69.0,AAA,2013J,TMA,19.0,10.0
4,1752,38053,19,0,79.0,AAA,2013J,TMA,19.0,10.0


In [6]:
# Keep only assessments from first 30 days
assessment_data = assessment_data[
    assessment_data["date"] <= 30
]

assessment_data.head()

,id_assessment,id_student,date_submitted,is_banked,score,code_module,code_presentation,assessment_type,date,weight
0,1752,11391,18,0,78.0,AAA,2013J,TMA,19.0,10.0
1,1752,28400,22,0,70.0,AAA,2013J,TMA,19.0,10.0
2,1752,31604,17,0,72.0,AAA,2013J,TMA,19.0,10.0
3,1752,32885,26,0,69.0,AAA,2013J,TMA,19.0,10.0
4,1752,38053,19,0,79.0,AAA,2013J,TMA,19.0,10.0


In [7]:
# Create Average Quiz Score Feature
quiz_score = assessment_data.groupby(
    ["code_module","code_presentation","id_student"]
)["score"].mean().reset_index()

quiz_score.rename(
    columns={"score":"avg_quiz_score"},
    inplace=True
)

quiz_score.head()

,code_module,code_presentation,id_student,avg_quiz_score
0,AAA,2013J,11391,78.0
1,AAA,2013J,28400,70.0
2,AAA,2013J,31604,72.0
3,AAA,2013J,32885,69.0
4,AAA,2013J,38053,79.0


In [8]:
# Assessment Count
assessment_count = assessment_data.groupby(
    ["code_module","code_presentation","id_student"]
).size().reset_index(name="assessments_completed")

assessment_count.head()

,code_module,code_presentation,id_student,assessments_completed
0,AAA,2013J,11391,1
1,AAA,2013J,28400,1
2,AAA,2013J,31604,1
3,AAA,2013J,32885,1
4,AAA,2013J,38053,1


In [9]:
# Average Assessment Weight
assessment_weight = assessment_data.groupby(
    ["code_module","code_presentation","id_student"]
)["weight"].mean().reset_index()

assessment_weight.rename(
    columns={"weight":"avg_assessment_weight"},
    inplace=True
)

assessment_weight.head()

,code_module,code_presentation,id_student,avg_assessment_weight
0,AAA,2013J,11391,10.0
1,AAA,2013J,28400,10.0
2,AAA,2013J,31604,10.0
3,AAA,2013J,32885,10.0
4,AAA,2013J,38053,10.0


In [10]:
# Average Submission Day
submission_day = assessment_data.groupby(
    ["code_module","code_presentation","id_student"]
)["date_submitted"].mean().reset_index()

submission_day.rename(
    columns={"date_submitted":"avg_submission_day"},
    inplace=True
)

submission_day.head()

,code_module,code_presentation,id_student,avg_submission_day
0,AAA,2013J,11391,18.0
1,AAA,2013J,28400,22.0
2,AAA,2013J,31604,17.0
3,AAA,2013J,32885,26.0
4,AAA,2013J,38053,19.0


In [11]:
# Video Engagement
video_engagement = studentVle.groupby(
    ["code_module","code_presentation","id_student"]
)["sum_click"].sum().reset_index()

video_engagement.rename(
    columns={"sum_click":"video_clicks"},
    inplace=True
)

video_engagement.head()

,code_module,code_presentation,id_student,video_clicks
0,AAA,2013J,11391,934
1,AAA,2013J,28400,1435
2,AAA,2013J,30268,281
3,AAA,2013J,31604,2158
4,AAA,2013J,32885,1034


In [12]:
# Login Frequency
login_frequency = studentVle.groupby(
    ["code_module","code_presentation","id_student"]
).size().reset_index(name="login_frequency")

login_frequency.head()

,code_module,code_presentation,id_student,login_frequency
0,AAA,2013J,11391,196
1,AAA,2013J,28400,430
2,AAA,2013J,30268,76
3,AAA,2013J,31604,663
4,AAA,2013J,32885,352


In [13]:
# Average Daily Activity
daily_activity = studentVle.groupby(
    ["code_module","code_presentation","id_student"]
)["date"].mean().reset_index()

daily_activity.rename(
    columns={"date":"avg_activity_day"},
    inplace=True
)

daily_activity.head()

,code_module,code_presentation,id_student,avg_activity_day
0,AAA,2013J,11391,102.132653
1,AAA,2013J,28400,86.993023
2,AAA,2013J,30268,2.355263
3,AAA,2013J,31604,106.147813
4,AAA,2013J,32885,91.934659


In [14]:
# Registration Duration
studentRegistration["registration_days"] = np.where(
    studentRegistration["date_unregistration"]==-1,
    270,
    studentRegistration["date_unregistration"]-
    studentRegistration["date_registration"]
)

studentRegistration.head()

,code_module,code_presentation,id_student,date_registration,date_unregistration,registration_days
0,AAA,2013J,11391,-159.0,-1.0,270.0
1,AAA,2013J,28400,-53.0,-1.0,270.0
2,AAA,2013J,30268,-92.0,12.0,104.0
3,AAA,2013J,31604,-52.0,-1.0,270.0
4,AAA,2013J,32885,-176.0,-1.0,270.0


In [15]:
# Merge Every Feature
final_data = studentInfo.copy()

final_data = final_data.merge(
    quiz_score,
    on=["code_module","code_presentation","id_student"],
    how="left"
)

final_data = final_data.merge(
    assessment_count,
    on=["code_module","code_presentation","id_student"],
    how="left"
)

final_data = final_data.merge(
    assessment_weight,
    on=["code_module","code_presentation","id_student"],
    how="left"
)

final_data = final_data.merge(
    submission_day,
    on=["code_module","code_presentation","id_student"],
    how="left"
)

final_data = final_data.merge(
    video_engagement,
    on=["code_module","code_presentation","id_student"],
    how="left"
)

final_data = final_data.merge(
    login_frequency,
    on=["code_module","code_presentation","id_student"],
    how="left"
)

final_data = final_data.merge(
    daily_activity,
    on=["code_module","code_presentation","id_student"],
    how="left"
)

final_data = final_data.merge(
    studentRegistration[
        [
            "code_module",
            "code_presentation",
            "id_student",
            "registration_days"
        ]
    ],
    on=["code_module","code_presentation","id_student"],
    how="left"
)

In [16]:
# Fill Missing Values
final_data.fillna(0,inplace=True)

,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,...,final_result,dropout,avg_quiz_score,assessments_completed,avg_assessment_weight,avg_submission_day,video_clicks,login_frequency,avg_activity_day,registration_days
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,...,Pass,0,78.0,1.0,10.0,18.0,934.0,196.0,102.132653,270.0
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,...,Pass,0,70.0,1.0,10.0,22.0,1435.0,430.0,86.993023,270.0
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,...,Withdrawn,1,0.0,0.0,0.0,0.0,281.0,76.0,2.355263,104.0
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,...,Pass,0,72.0,1.0,10.0,17.0,2158.0,663.0,106.147813,270.0
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,...,Pass,0,69.0,1.0,10.0,26.0,1034.0,352.0,91.934659,270.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32588,GGG,2014J,2640965,F,Wales,Lower Than A Level,10-20,0-35,0,30,...,Fail,0,0.0,0.0,0.0,0.0,41.0,19.0,14.368421,270.0
32589,GGG,2014J,2645731,F,East Anglian Region,Lower Than A Level,40-50%,35-55,0,30,...,Distinction,0,0.0,0.0,0.0,0.0,893.0,237.0,142.928270,270.0
32590,GGG,2014J,2648187,F,South Region,A Level or Equivalent,20-30%,0-35,0,30,...,Pass,0,0.0,0.0,0.0,0.0,312.0,108.0,128.981481,270.0
32591,GGG,2014J,2679821,F,South East Region,Lower Than A Level,90-100%,35-55,0,30,...,Withdrawn,1,0.0,0.0,0.0,0.0,275.0,61.0,36.721311,150.0


In [17]:
# Check Dataset
print(final_data.shape)

final_data.head()

(32593, 21)


,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,...,final_result,dropout,avg_quiz_score,assessments_completed,avg_assessment_weight,avg_submission_day,video_clicks,login_frequency,avg_activity_day,registration_days
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,...,Pass,0,78.0,1.0,10.0,18.0,934.0,196.0,102.132653,270.0
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,...,Pass,0,70.0,1.0,10.0,22.0,1435.0,430.0,86.993023,270.0
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,...,Withdrawn,1,0.0,0.0,0.0,0.0,281.0,76.0,2.355263,104.0
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,...,Pass,0,72.0,1.0,10.0,17.0,2158.0,663.0,106.147813,270.0
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,...,Pass,0,69.0,1.0,10.0,26.0,1034.0,352.0,91.934659,270.0


In [18]:
# Missing Values
final_data.isnull().sum()


code_module              0
code_presentation        0
id_student               0
gender                   0
region                   0
highest_education        0
imd_band                 0
age_band                 0
num_of_prev_attempts     0
studied_credits          0
disability               0
final_result             0
dropout                  0
avg_quiz_score           0
assessments_completed    0
avg_assessment_weight    0
avg_submission_day       0
video_clicks             0
login_frequency          0
avg_activity_day         0
registration_days        0
dtype: int64

In [19]:
# Drop Unnecessary Columns
drop_columns = [
    "id_student"
]

final_data.drop(columns=drop_columns,inplace=True)

In [20]:
# Final Columns
print(final_data.columns)

Index(['code_module', 'code_presentation', 'gender', 'region',
       'highest_education', 'imd_band', 'age_band', 'num_of_prev_attempts',
       'studied_credits', 'disability', 'final_result', 'dropout',
       'avg_quiz_score', 'assessments_completed', 'avg_assessment_weight',
       'avg_submission_day', 'video_clicks', 'login_frequency',
       'avg_activity_day', 'registration_days'],
      dtype='str')


In [21]:
# Save Final Dataset
output_path = "../Dataset/final_dataset.csv"

final_data.to_csv(
    output_path,
    index=False
)

print("Final dataset created successfully")

Final dataset created successfully


In [22]:
# Verify
df = pd.read_csv("../Dataset/final_dataset.csv")

df.head()

,code_module,code_presentation,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,dropout,avg_quiz_score,assessments_completed,avg_assessment_weight,avg_submission_day,video_clicks,login_frequency,avg_activity_day,registration_days
0,AAA,2013J,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass,0,78.0,1.0,10.0,18.0,934.0,196.0,102.132653,270.0
1,AAA,2013J,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,Pass,0,70.0,1.0,10.0,22.0,1435.0,430.0,86.993023,270.0
2,AAA,2013J,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,Y,Withdrawn,1,0.0,0.0,0.0,0.0,281.0,76.0,2.355263,104.0
3,AAA,2013J,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,N,Pass,0,72.0,1.0,10.0,17.0,2158.0,663.0,106.147813,270.0
4,AAA,2013J,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,N,Pass,0,69.0,1.0,10.0,26.0,1034.0,352.0,91.934659,270.0
